In [3]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append(f"./../")

In [8]:
from src.graphs import StaticGraph
from src.misc import graph_to_bitstring_edges
import networkx as nx
import numpy as np
import itertools

In [13]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Pauli

# Create your quantum circuit
qc = QuantumCircuit(3)
qc.x(0)
qc.y(1)
qc.z(2)

# Convert the circuit to a Pauli operator
pauli_op = Pauli(qc)

# Get the Pauli string
pauli_string = str(pauli_op)

print(pauli_string)

ZYX


In [6]:
graph_edges =[]
g6 = '../data/graphs/negative_graphs.g6'

with open(g6, 'r') as file:
    for line in file:
        g6_string = line.strip()
        graph = nx.from_graph6_bytes(g6_string.encode('utf-8'))
        bitstring_edges = graph_to_bitstring_edges(graph)
        print(bitstring_edges)


{('001', '010'), ('001', '111'), ('010', '110'), ('000', '001'), ('001', '101'), ('010', '100'), ('001', '011')}
{('001', '010'), ('001', '111'), ('001', '110'), ('000', '001'), ('100', '101'), ('010', '101'), ('001', '011')}
{('001', '010'), ('010', '111'), ('001', '110'), ('010', '011'), ('000', '001'), ('010', '101'), ('001', '100')}
{('001', '010'), ('001', '110'), ('000', '001'), ('011', '100'), ('001', '101'), ('010', '100'), ('100', '111')}
{('001', '010'), ('001', '110'), ('000', '001'), ('011', '100'), ('011', '111'), ('001', '101'), ('010', '011'), ('001', '011')}
{('001', '010'), ('001', '110'), ('000', '001'), ('011', '100'), ('001', '101'), ('010', '100'), ('100', '111'), ('001', '011')}
{('001', '010'), ('001', '110'), ('010', '011'), ('000', '001'), ('011', '100'), ('011', '111'), ('001', '101'), ('001', '100'), ('001', '011')}
{('001', '010'), ('001', '111'), ('001', '110'), ('011', '101'), ('000', '001'), ('100', '101'), ('010', '101'), ('001', '100'), ('001', '011')}


In [23]:
import networkx as nx
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Pauli
from src.graphs import StaticGraph, DynamicGraph, IntersectingEdgesGraph, MultiEdgeGraph
import numpy as np
import itertools

def graph_to_bitstring_edges(graph):
    num_nodes = len(graph.nodes)
    num_bits = len(bin(num_nodes - 1)) - 2
    node_to_bitstring = {node: format(node, f'0{num_bits}b') for node in graph.nodes}
    edges_bitstring = {(node_to_bitstring[u], node_to_bitstring[v]) for u, v in graph.edges}
    return edges_bitstring

def unitary_to_pauli(U):
    n = int(np.log2(U.shape[0]))
    dim = 2**n
    pauli_strings = []
    coeffs = []
    for pauli_string in [''.join(p) for p in itertools.product('IXYZ', repeat=n)]:
        P = Pauli(pauli_string)
        P_op = Operator(P).data
        coeff = np.trace(P_op.conj().T @ U) / dim
        if not np.isclose(coeff, 0, atol=1e-10):
            pauli_strings.append(pauli_string)
            coeffs.append(coeff)
    return SparsePauliOp(pauli_strings, coeffs)

def get_dynamic_walk_circuit(edges, T, delta_t):
    G = StaticGraph(edges)
    graph_sequence = [(G, T)]
    dyn_G = DynamicGraph(graph_sequence)
    intersecting_G = IntersectingEdgesGraph(edges)
    graph_sequence = [(graph, delta_t) for graph in intersecting_G.subgraphs]
    dyn_G_approx = DynamicGraph(graph_sequence)
    
    big_qc = QuantumCircuit(G.n_qubits)
    for sq in dyn_G_approx.graph_sequence:
        G = MultiEdgeGraph(sq[0].edges)
        sub_qc = G.get_qc(simplified=True)
        big_qc = big_qc.compose(sub_qc)
    transpiled_qc = transpile(big_qc, basis_gates=['cx', 'u3'], optimization_level=3)
    return transpiled_qc

def circuit_to_pauli_decomposition(circuit):
    op = Operator(circuit)
    unitary = op.data
    return unitary_to_pauli(unitary)

def compact_pauli_string(sparse_pauli_op):
    terms = []
    for pauli_string, coeff in zip(sparse_pauli_op.paulis.to_labels(), sparse_pauli_op.coeffs):
        if np.abs(coeff) > 1e-10:  # Ignore very small coefficients
            coeff_str = f"{coeff.real:.6f}".rstrip('0').rstrip('.')
            if coeff.imag != 0:
                coeff_str += f"{coeff.imag:+.6f}j".rstrip('0').rstrip('.')
            
            terms.append(f"{coeff_str}{pauli_string}")

    return ' + '.join(terms)


input_file = '../data/graphs/negative_graphs.g6'
output_file = '../data/graphs/negative_graphs_analysis.txt'

T = 200
delta_t = 0.1

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for i, line in enumerate(infile):
        g6_string = line.strip()
        graph = nx.from_graph6_bytes(g6_string.encode('utf-8'))
        bitstring_edges = graph_to_bitstring_edges(graph)
        
        # Get the Pauli string from unitary_to_pauli
        G = StaticGraph(bitstring_edges)
        pauli_op = unitary_to_pauli(G.get_adj_mat())
        pauli_string_unitary = compact_pauli_string(pauli_op)
        
        # Get the Pauli string for the circuit from get_dynamic_walk_circuit
        dynamic_circuit = get_dynamic_walk_circuit(bitstring_edges, T, delta_t)
        pauli_op_dynamic = circuit_to_pauli_decomposition(dynamic_circuit)
        pauli_string_dynamic = compact_pauli_string(pauli_op_dynamic)
        
        # Write the information to the output file
        outfile.write(f"Graph {i+1}:{bitstring_edges}\n")
        outfile.write(f"Pauli Decompositions: {pauli_string_unitary}\n")
        outfile.write(f"Dynamic Quantum Walk: {pauli_string_dynamic}\n")
        outfile.write("----------------------------------------------------------------------------------------\n")

print(f"Analysis completed. Results written to {output_file}")

Analysis completed. Results written to ../data/graphs/negative_graphs_analysis.txt
